# L4a: Trading Rules Using Binomial Lattice Models
In this lecture, we continue our exploration of binomial lattice models of share price dynamics. We will derive trade entry and exit rules based on the net present value (NPV) of long positions in a stock. This is our first trading-focused lecture!

> __Learning Objectives__
>
> By the end of this lecture, you will be able to:
> 
> * **Develop NPV-based trade entry and exit rules using binomial lattice models.** We will derive expressions for the net present value of long equity positions and show how to compute the probability of achieving target returns or losses over different holding periods.
> * **Calculate cumulative probabilities for profit and loss thresholds using lattice models.** Using closed-form expressions, we will determine the likelihood of exceeding target fractional returns or breaching stop-loss levels based on the binomial distribution of up-moves.
> * **Extend binomial models to N-ary lattice frameworks for richer price dynamics.** We will generalize from two-state (up/down) models to multi-state recombining trees that can approximate continuous price distributions in the limit of many possible outcomes.

This is going to be a fun lecture! Let's get started!
___

## Examples
Today, we will use two examples to connect the lecture's probability formulas to data, code and decision-making:

> [▶ Explore a terminal target probability](CHEME-5660-L4a-Example-CumulativeProbabilityLattice-Fall-2026.ipynb). In this example, we will test the binomial lattice model trade exit rules. We will use a binomial lattice model to compute the cumulative probability of achieving at least a target fractional return (ROI) of $r_{\star}$ (or alternatively, a target fractional loss of $-r_{\star}$) over a holding period.

> [▶ Explore N-ary lattice models](CHEME-5660-L4a-Example-N-Ary-Lattice-Fall-2026.ipynb). In this example, we extend the binomial lattice model to an n-ary lattice model, where the share price can move to many possible values at each time step (not just up or down). In the limit of many possible values, we can approximate a continuous distribution of share prices. Wow!

Optional extensions are collected near the end of the lecture.

___

## From the Bid–Ask Spread to a Lattice Price
The [L3a lecture](../../week-3/L3a/CHEME-5660-L3a-Lecture-Equity-Exchanges-Return-StylizedFacts-Fall-2026.ipynb) introduced exchanges, order types, order books, and the National Best Bid and Offer (NBBO). We need only one piece of that machinery today: the price used in a forecast is not automatically the price at which a trade executes.

__Market makers, Citadel Securities and the spread.__ [Citadel Securities](https://www.citadelsecurities.com) is a market maker. At time $t$, a market maker may quote a bid $b_t$, the price at which it will buy, and an ask $a_t$, the price at which it will sell. The bid–ask spread $s_{t}$ and midpoint $m_{t}$ are given by:
$$
s_t=a_t-b_t,\qquad m_t=\frac{a_t+b_t}{2}.
$$

In the NPV model we introduce in lecture, we will replace the executable bid and ask quotes $(b_t,a_t)$ with a single frictionless price $S_t$. The [execution-aware extension](advanced/execution/CHEME-5660-L4a-Advanced-ExecutionAware-ProbabilityOfProfit-Fall-2026.ipynb) restores the spread, fees, and slippage.

Given these simplifying assumptions, let's specify the probability model for hitting a specific exit price at time $t$, denoted by $S_t$.

### Review: The Binomial Lattice Model
A binomial lattice represents an asset price on a discrete time grid. From each state, the next price moves by one of two multiplicative factors.

<div>
    <center>
        <img src="figs/Fig-Lattice-Schematic.svg" width="800" alt="Three panels showing one-, two-, and three-step binomial lattices in which up and down branches recombine"/>
    </center>
</div>

> __What distribution does a constant-parameter binomial lattice imply?__
>
> Let $S_0>0$ be the initial price (root of the tree). At each step, we flip a coin, i.e., we perform a Bernoulli trial, and the price either goes up by a factor $u$ with probability $p$ or by down by factor $d$ with probability $1-p$, where $u>d>0$ and $0<p<1$. Holding $(u,d,p)$ fixed across time makes the moves independent and identically distributed.
>
> If $K_t$ is the number of up moves in the first $t$ steps, then $K_t\sim\operatorname{Binomial}(t,p)$. Thus, the induced probability of each level-$t$ price node is described by a Binomial Distribution:
> $$
\boxed{
> \mathbb P\!\left(S_t=S_0u^kd^{t-k}\right)
> =\binom{t}{k}p^k(1-p)^{t-k},
> \qquad k=0,1,\ldots,t.}
> $$
> Level $t$ of the tree contains $t+1$ share price states, i.e., a complete finite distribution for the share price at time $t$. Different assets or calibration windows may produce different values of $(u,d,p)$.

Because this model uses constant parameters and independent moves, it cannot reproduce heavy tails or volatility clustering. Later in the course, related lattices will price derivatives rather than forecast share prices where we use the risk-neutral probability measure $\mathbb{Q}$ derived in [L3b lecture](../../week-3/L3b/CHEME-5660-L3b-Lecture-LatticeModels-Equity-Price-Fall-2026.ipynb).

___

## Theory: Net Present Value (NPV) Trade Rule
Suppose we buy $n_0>0$ shares of `XYZ` at time $0$ for $S_0$ USD/share and sell all the `XYZ` shares at $T=N\Delta t$ for $S_T$ USD/share. Let's assume a frictionless price return with no dividends. Finally, following [the L1b lecture](../../week-1/L1b/CHEME-5660-L1b-Lecture-TimeValueMoney-Fall-2026.ipynb), let $g_y$ denote the continuously compounded annual risk-fee growth rate associated with the selected benchmark yield $y$ (units: inverse years). 

<div>
    <center>
        <img src="figs/Fig-TradeRule-Schematic.svg" width="800" alt="Discounted per-share NPV versus terminal share price; the entry price lies below zero NPV and a higher break-even threshold separates loss and profit regions"/>
    </center>
</div>


 > __What return does the discounted cash-flow rule measure?__
>
> Let's use our abstract asset framework to compute trade-exit rules. From the investor's perspective, the entry cash flow, i.e., when we purchase the shares is $-n_0S_0$ USD at time $0$ and the discounted exit cash flow (when we sell the shares) is $n_0S_Te^{-g_yT}$ USD. Therefore, the scaled net present value (NPV) for this trade $\rho_T$ is given by:
> $$
> \begin{align*}
> \operatorname{NPV}(g_y,T)
> &= \underbrace{-n_0S_0}_{\text{entry}}+\underbrace{n_0S_Te^{-g_yT}}_{\text{exit}},\quad\text{divide by initial $n_{0}S_{0}$}\\
> \rho_T
> \equiv \underbrace{\frac{\operatorname{NPV}(g_y,T)}{n_0S_0}}_{\text{discounted fractional return}}
> &=\left(\frac{S_T}{S_0}\right)e^{-g_yT}-1.
> \end{align*}
> $$
> The scaled NPV $\rho_T$ is a dimensionless present-value return on the initial share cost. It is positive exactly when the discounted sale value exceeds the entry cost. A probability model for $S_T$ therefore induces a probability distribution for $\rho_T$, i.e., we can use a model for $S_{T}$ to compute the probability of a specified $\rho_T$ level.

__Short-horizon approximation.__ When $|g_y|T$ is small, $e^{-g_yT}\approx1$ and $\rho_T\approx S_T/S_0-1$. Thus, on a short time scale the scaled NPV is the __fractional return__. However, the exact discounted expression remains our default.

___

## Terminal Target Probability
Suppose we fix a scheduled exit at $T=N\Delta t$ and a target scaled NPV $\rho_\star$. The event is strict: a terminal node whose return equals the target does not count as success. Because the terminal return is monotone in the binomial up-move count, the entire event can be characterized by one integer cutoff.

> __Terminal-Target Theorem (binomial lattice):__
>
> Let $N\in\mathbb Z_{\geq 1}$ be the number of lattice steps, let $\Delta t>0$ be the duration of each step in years, and let $T=N\Delta t$ be the scheduled holding period. Let $S_0>0$ be the initial asset price; let $u>d>0$ be the multiplicative up- and down-move factors; and let $p\in(0,1)$ be the probability of an up move, so the probability of a down move is $1-p$. Finally, let $g_y\in\mathbb R$ be the continuously compounded annual growth rate associated with the selected benchmark yield $y$. 
> 
> Suppose the lattice moves are independent and identically distributed. Then $K_N$, the number of up moves during the $N$ steps, satisfies $K_N\sim\operatorname{Binomial}(N,p)$. The scaled NPV at the scheduled exit is given by:
> $$
> \rho_T=\rho(K_N)
> =u^{K_N}d^{N-K_N}e^{-g_yN\Delta t}-1.
> $$
> For any target scaled NPV $\rho_\star\in\mathbb R$, the success event $\{\rho_T>\rho_\star\}$ is a binomial upper tail. If $\rho_\star\leq-1$, then the success probability is given by:
> $$
> \mathbb P(\rho_T>\rho_\star)=1,
> $$
> because every attainable frictionless terminal return is strictly greater than $-1$.
>
> __Derivation for $\rho_\star>-1$.__ Since $u/d>1$, the terminal return increases strictly with $K_N$. Therefore, the strict target event can be rewritten as follows:
> $$
> \begin{align*}
> \rho(K_N)>\rho_\star
> &\iff u^{K_N}d^{N-K_N}e^{-g_yN\Delta t}>1+\rho_\star,\\
> &\iff K_N\ln(u/d)>
> \ln(1+\rho_\star)+g_yN\Delta t-N\ln d,\\
> &\iff K_N>\tau(\rho_\star).
> \end{align*}
> $$
> Define the real-valued threshold by:
> $$
> \tau(\rho_\star)=
> \frac{\ln(1+\rho_\star)+g_yN\Delta t-N\ln d}{\ln(u/d)}.
> $$
> The strict integer cutoff is $\kappa=\lfloor\tau(\rho_\star)\rfloor+1$. Clipping low cutoffs to $0$ and using $N+1$ as the infeasible-event sentinel gives:
> $$
> k_{\min}=\min\!\left\{\max\!\left\{\kappa,0\right\},N+1\right\}.
> $$
> Therefore, for $\rho_\star>-1$, the binomial upper-tail probability is given by:
> $$
> \mathbb P(\rho_T>\rho_\star)
> =\mathbb P(K_N\geq k_{\min})
> =\begin{cases}
> 1, & k_{\min}=0,\\
> \displaystyle\sum_{k=k_{\min}}^N\binom Nk p^k(1-p)^{N-k},
>     & 1\leq k_{\min}\leq N,\\
> 0, & k_{\min}=N+1.
> \end{cases}
> $$
> If $\tau=q$ is an attainable integer, the node $K_N=q$ reaches the target exactly and is excluded; strict success begins at $q+1$. Thus, the inequality determines the cutoff, while $p$ determines the probability mass above it. In floating-point code, checking the actual node returns is safer than relying only on `floor(τ)`, which can misclassify an equality boundary after roundoff.

The complementary probability is $\mathbb P(\rho_T\leq\rho_\star)=1-\mathbb P(\rho_T>\rho_\star)$. Increasing $g_y$ at fixed $N$ raises the hurdle, but changing the horizon changes both the cutoff and the distribution, so the full probability must be recomputed.

Let's explore these ideas with an example.

__Example__:

> [▶ Explore a terminal target probability](CHEME-5660-L4a-Example-CumulativeProbabilityLattice-Fall-2026.ipynb). In this example, we will test the binomial lattice model trade exit rules. We will use a binomial lattice model to compute the cumulative probability of achieving at least a target fractional return (ROI) of $r_{\star}$ (or alternatively, a target fractional loss of $-r_{\star}$) over a holding period.

<!-- The companion [terminal-target example](CHEME-5660-L4a-Example-CumulativeProbabilityLattice-Fall-2026.ipynb) estimates $(u,d,p)$ from data, finds the first node that satisfies the strict return predicate, and evaluates the corresponding binomial survival probability. -->

___

## Terminal Rules and First-Passage Rules Are Different
The binomial tail above checks the position once, at the scheduled date $T$. A take-profit or stop-loss rule monitored along the way asks which boundary is detected first and is therefore path dependent.

> __What state is needed for a monitored exit rule?__
>
> Let $U>S_0$ be a take-profit boundary and $L<S_0$ a stop-loss boundary. On the lattice grid, define the first-hitting times as:
> $$
> \tau_U=\min\{j\ge1:S_j\ge U\},
> \qquad
> \tau_L=\min\{j\ge1:S_j\le L\}.
> $$
> With the minimum of an empty set defined as $\infty$, a monitored rule distinguishes $\{\tau_U\le N,\ \tau_U<\tau_L\}$, $\{\tau_L\le N,\ \tau_L<\tau_U\}$, and the open event $\{\min(\tau_U,\tau_L)>N\}$. Two paths can finish at the same terminal node while belonging to different events. An absorbing recursion therefore propagates only the probability mass that remains open and records mass when it first reaches either boundary.

Execution remains part of the rule: a triggered stop becomes a market order and may not fill at its trigger price, whereas a limit order constrains price but may not fill. The optional [first-passage notebook](advanced/first-passage/CHEME-5660-L4a-Advanced-FirstPassage-ExitRules-Fall-2026.ipynb) implements the absorbing recursion, and the [execution-aware notebook](advanced/execution/CHEME-5660-L4a-Advanced-ExecutionAware-ProbabilityOfProfit-Fall-2026.ipynb) restores spread, fees, and slippage.

We now return to terminal distributions and ask how the state space changes when each step has more than two outcomes.

___

## N-Ary Lattice Models
One obvious limitation of the binomial lattice model is that it only allows for two possible price movements at each time step (up or down). In reality, asset prices can exhibit a wider range of behaviors. 

> __Idea__: If two possible futures are'nt enough, then let's extend the binomial lattice model to an N-ary model, e.g., a trinomial lattice model (three possible price movements: up, down, or unchanged) or even more complex models with multiple price levels at each time step. Let's explore this idea.

Today, we are going to limit ourselves to __recombining trees__. A recombining N-ary tree models $n$ possible future states per step but merges paths that arrive at the same state. 

> __How are the states, prices, probabilities, and storage counts connected?__
>
> Assume branch choices are independent across steps and use fixed factors and probabilities. At level $t$, let $\mathbf x=(x_1,\ldots,x_m)$ record how many times each branch occurred, with $x_j\ge0$ and $\sum_jx_j=t$. The price and probability associated with this branch-count state are given by:
> $$
> \begin{align*}
> S_t(\mathbf x)&=S_0\prod_{j=1}^m f_j^{x_j},\\
> \mathbb P(\mathbf X_t=\mathbf x)
> &=\frac{t!}{x_1!\cdots x_m!}\prod_{j=1}^m p_j^{x_j}.
> \end{align*}
> $$
> Stars and bars gives the number of branch-count states at level $t$, the total through height $h$, and the zero-based flat-array offset of level $t$:
> $$
> L_t=\binom{t+m-1}{m-1},
> \qquad
> \sum_{r=0}^hL_r=\binom{h+m}{m},
> \qquad
> O(t)=\sum_{r=0}^{t-1}L_r=\binom{t+m-1}{m}.
> $$
> These are counts of branch-count states. If different factor products coincide, distinct count states can share the same numerical price. The formulas still determine how much state must be stored and how multinomial probability mass is assigned.

More branches give a finer discrete approximation to a one-step distribution, but they do not guarantee a better forecast; the binning and parameter estimates still require validation. 

Let's explore these ideas with an example.

__Example__:

> [▶ Explore N-ary lattice models](CHEME-5660-L4a-Example-N-Ary-Lattice-Fall-2026.ipynb). In this example, we extend the binomial lattice model to an n-ary lattice model, where the share price can move to many possible values at each time step (not just up or down). In the limit of many possible values, we can approximate a continuous distribution of share prices. Wow!


With suitable scaling as the time step shrinks, a lattice can converge to a continuous stochastic price model. That is the starting point for L4b.
___

## Optional Advanced Material
The notebooks below extend today's material. They are optional and are not prerequisites for L4b; the [advanced index](advanced/README.md) lists them with a suggested order.

* [▶ First-passage exit rules](advanced/first-passage/CHEME-5660-L4a-Advanced-FirstPassage-ExitRules-Fall-2026.ipynb). Compute exact take-profit and stop-loss first-passage probabilities by propagating only the probability mass for positions that remain open.
* [▶ Execution-aware probability of profit](advanced/execution/CHEME-5660-L4a-Advanced-ExecutionAware-ProbabilityOfProfit-Fall-2026.ipynb). Add the bid-ask spread, fees, and slippage to the terminal probability-of-profit calculation and compare with the frictionless rule.

___

## Summary
In this lecture, we converted lattice price distributions into precisely defined terminal target events and then extended the state space beyond two branches.

> __Key Takeaways:__
>
> * **A terminal target becomes a binomial tail:** Once the entry price, terminal exit, horizon, benchmark, cost assumptions, and strictness fully specify the rule, the monotone terminal return converts the target into a minimum up-move count whose probability is a binomial tail, with infeasible thresholds handled explicitly.
>
> * **Monitored exits are path dependent:** A take-profit or stop-loss rule checked during the holding period is a first-passage problem, not a terminal-node event.
>
> * **N-ary states use branch counts:** A multinomial count vector determines each recombining node's price and probability, and the number of states grows with the branch count and horizon.

Next time: Continuous stochastic models of price dynamics.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products, or any investment or trading advice or strategy, is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.